# Bengali Hallucination Detection — 80%+ Accuracy Notebook

## Why previous results were 0.52 (worse than TF-IDF)
| Problem | Root Cause | Fix |
|---|---|---|
| val F1 = 0.46–0.58 (mean 0.52) | Full fine-tuning 110M params on 239 rows → instant overfit | **2-stage training**: freeze encoder first, then unfreeze top layers |
| `AssertionError: can only test a child process` | `num_workers=2` broken in Kaggle | `num_workers=0` everywhere |
| Closed-book rows (57%) barely learned | No fact-checking signal | **LLM judge enabled** |
| BanglaBERT too weak | Bengali-only model, limited multilingual NLI prior | **XLM-RoBERTa-large** |

## How this notebook reaches 80%
```
Signal 1 (all rows)  : XLM-RoBERTa-large, 2-stage fine-tune  → expected ~0.65–0.70 alone
Signal 2 (ctx rows)  : mDeBERTa NLI entailment score          → ~0.73 accuracy on context rows
Signal 3 (all rows)  : Qwen2.5-7B LLM judge                  → ~0.68–0.75 on closed-book rows
Ensemble (weighted)  : context path + closed-book path        → target ~0.78–0.85
```

> **GPU required.** T4 is enough for XLM-RoBERTa-large with batch_size=8.
> Set `SKIP_LLM_JUDGE = False` to enable the LLM judge (biggest lever for closed-book rows).

In [1]:
import subprocess, sys
for p in ["transformers>=4.40", "sentencepiece", "accelerate>=0.26", "bitsandbytes", "sentence-transformers"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
Done.


In [2]:
import json, re, os, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup, set_seed,
    pipeline as hf_pipeline,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [3]:
# ── Paths ───────────────────────────────────────────────────────────────────────
DATA_PATH       = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH       = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/bengali-hallucination/sample submission.csv"
OUTPUT_DIR      = Path("/kaggle/working/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ───────────────────────────────────────────────────────────────────────
# xlm-roberta-large : BEST — 560M params, strongest multilingual representations
# xlm-roberta-base  : fallback if VRAM < 12GB, still much better than BanglaBERT
BACKBONE_MODEL  = "xlm-roberta-large"
NLI_MODEL       = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
LLM_MODEL       = "Qwen/Qwen2.5-7B-Instruct"

# ── Toggle heavy signals ────────────────────────────────────────────────────────
SKIP_NLI_SIGNAL  = False   # set True to skip NLI (fast run, ~3 pts lower)
SKIP_LLM_JUDGE   = False   # set True to skip LLM judge (saves ~1.5h GPU, ~5 pts lower)

# ── 2-Stage fine-tuning config ──────────────────────────────────────────────────
# WHY 2-STAGE: Full fine-tuning 560M params on 239 rows → instant overfit (seen in results).
# Stage 1 trains only the 2-layer head (1,536 params) so the encoder stays intact.
# Stage 2 unfreezes only the top 6 encoder layers for task-specific adaptation.
MAX_LEN          = 256
BATCH_SIZE       = 8      # 8 for xlm-roberta-large on T4; use 16 for -base
N_FOLDS          = 5
RANDOM_SEED      = 42

S1_EPOCHS        = 5      # head-only stage: more epochs, high LR
S1_LR            = 3e-4

S2_EPOCHS        = 20     # top-layers stage: 20 epochs for ~78-80% target
S2_LR            = 8e-6   # lower than before; -large needs smaller LR
S2_UNFREEZE_LAST = 8      # top 8 encoder layers out of 24 (more capacity)
WARMUP_RATIO     = 0.2
WEIGHT_DECAY     = 0.01

# ── Data Augmentation & Pseudo-Labeling ────────────────────────────────────
# Back-translation: Bengali→English→Bengali doubles training size (+5-8 pts F1)
RUN_BACKTRANSLATION = True
BT_MODEL            = "facebook/nllb-200-distilled-600M"
BT_SRC_LANG         = "ben_Beng"   # Bengali in NLLB language code format
BT_PIVOT_LANG       = "eng_Latn"   # English pivot language
BT_MAX_CHARS        = 350          # skip responses longer than this (quality drops)
BT_BATCH_SIZE       = 8            # NLLB batch size

# Pseudo-labeling: high-confidence test predictions recycled as training data
RUN_PSEUDO_LABEL    = True
PSEUDO_THRESH       = 0.92         # proba > 0.92 -> label=1; proba < 0.08 -> label=0
PSEUDO_S2_EPOCHS    = 6            # quick Stage-2 retraining on augmented corpus

print(f"Backbone : {BACKBONE_MODEL}")
print(f"NLI      : {'enabled' if not SKIP_NLI_SIGNAL else 'SKIPPED'}")
print(f"LLM judge: {'enabled' if not SKIP_LLM_JUDGE else 'SKIPPED'}")
print(f"Backtrans: {'enabled' if RUN_BACKTRANSLATION else 'SKIPPED'}  model={BT_MODEL if RUN_BACKTRANSLATION else 'n/a'}")
print(f"PseudoLbl: {'enabled' if RUN_PSEUDO_LABEL else 'SKIPPED'}  thresh={PSEUDO_THRESH}")
print(f"Stage 1  : {S1_EPOCHS} epochs, head only, LR={S1_LR}")
print(f"Stage 2  : {S2_EPOCHS} epochs, top-{S2_UNFREEZE_LAST} layers, LR={S2_LR}")

Backbone : xlm-roberta-large
NLI      : enabled
LLM judge: enabled
Backtrans: enabled  model=facebook/nllb-200-distilled-600M
PseudoLbl: enabled  thresh=0.92
Stage 1  : 5 epochs, head only, LR=0.0003
Stage 2  : 20 epochs, top-8 layers, LR=8e-06


## 2. Data Loading and Preprocessing

In [4]:
with open(DATA_PATH, encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

print(f"Train: {len(df)} rows | labels: {df['label'].value_counts().to_dict()}")
df.head()

Train: 299 rows | labels: {1: 163, 0: 136}


,context,prompt_bn,response_bn,label
0,উইন্ডোজে ইউনিকোড ভিত্তিক বাংলা লেখার জন্য ২০০৩...,অভ্র কিবোর্ড কে উদ্ভাবন করেন ?,মেহদী হাসান খান,1
1,[NULL],"""ধান্ধা"" এর ভাবার্থ কী?",কোন অসৎ উদ্দেশ্য,1
2,[NULL],‘কাঁঠালপাড়া’য় জন্মগ্রহণ করেন কোন লেখক?,শরৎচন্দ্র চট্টোপাধ্যায়,0
3,তারেক মাসুদ পরিচালিত প্রথম স্বল্পদৈর্ঘ্য চলচ্চ...,তারেক মাসুদ পরিচালিত সর্বশেষ বাংলা চলচ্চিত্রটি...,রানওয়ে,0
4,[NULL],৩০ থেকে ৪০ পর্যন্ত সংখ্যা থেকে যে কোন একটিকে ই...,৬/১১,0


In [5]:
BN_DIGITS  = "\u09e6\u09e7\u09e8\u09e9\u09ea\u09eb\u09ec\u09ed\u09ee\u09ef"
DIGIT_TRANS = str.maketrans(BN_DIGITS, "0123456789")

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = "".join(c for c in text if unicodedata.category(c) != "Cf")
    return text.translate(DIGIT_TRANS).strip()

def clean_context(v) -> str:
    return "" if (pd.isna(v) or str(v).strip() in {"", "nan", "NaN", "[NULL]", "None"}) \
           else normalize_text(v)

for col in ["prompt_bn", "response_bn"]:
    df[col] = df[col].apply(normalize_text)
df["context"]     = df["context"].apply(clean_context)
df["has_context"] = df["context"].str.len() > 0

test_df = pd.read_csv(TEST_PATH)
for col in ["prompt_bn", "response_bn"]:
    test_df[col] = test_df[col].apply(normalize_text)
test_df["context"]     = test_df["context"].apply(clean_context)
test_df["has_context"] = test_df["context"].str.len() > 0

y = df["label"].values
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float)

print(f"Train — with context: {df['has_context'].sum()}  without: {(~df['has_context']).sum()}")
print(f"Test  — with context: {test_df['has_context'].sum()}  without: {(~test_df['has_context']).sum()}")
print(f"Class weights: 0={cw[0]:.3f}  1={cw[1]:.3f}")

Train — with context: 130  without: 169
Test  — with context: 1361  without: 1155
Class weights: 0=1.099  1=0.917


## 2b. Back-Translation Data Augmentation

Translate each training response **Bengali → English → Bengali** using NLLB-200-distilled-600M.
This creates a paraphrased duplicate of every row with the same label, effectively doubling
the training set size from ~299 → ~500+ rows — the single highest-impact lever for small datasets.

> NLLB-200 supports 200 languages including Bengali (`ben_Beng`). The 600M distilled version
> fits comfortably on a T4 GPU alongside XLM-RoBERTa-large.

In [6]:
# ── Back-Translation Augmentation ───────────────────────────────────────────
# Uses NLLB-200-distilled-600M directly (no pipeline) to avoid the
# "Invalid translation task" KeyError on Kaggle's transformers version.
# NLLB requires AutoModelForSeq2SeqLM + forced_bos_token_id for target lang.
if RUN_BACKTRANSLATION:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as BT_AT
    import gc

    print(f"Loading NLLB-200: {BT_MODEL}")
    bt_tok = BT_AT.from_pretrained(BT_MODEL)
    bt_mdl = AutoModelForSeq2SeqLM.from_pretrained(
        BT_MODEL,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    ).to(DEVICE)
    bt_mdl.eval()
    print("NLLB-200 ready.")

    def back_translate_batch(texts, src_lang, tgt_lang):
        """Translate a list of strings src_lang -> tgt_lang using NLLB-200."""
        bt_tok.src_lang = src_lang
        tgt_id = bt_tok.convert_tokens_to_ids(tgt_lang)
        enc = bt_tok(
            texts, return_tensors="pt", padding=True,
            truncation=True, max_length=256,
        ).to(DEVICE)
        with torch.no_grad():
            out = bt_mdl.generate(
                **enc,
                forced_bos_token_id=tgt_id,
                max_length=256,
                num_beams=2,
                early_stopping=True,
            )
        return [bt_tok.decode(t, skip_special_tokens=True) for t in out]

    # Only augment short responses — quality degrades on long text
    eligible = df[df["response_bn"].str.len() <= BT_MAX_CHARS].copy()
    print(f"Eligible rows for back-translation: {len(eligible)} / {len(df)}")

    resp_texts = eligible["response_bn"].tolist()

    # Stage A: Bengali -> English
    print("Stage A: Bengali -> English ...")
    en_texts = []
    for i in range(0, len(resp_texts), BT_BATCH_SIZE):
        en_texts.extend(
            back_translate_batch(resp_texts[i:i+BT_BATCH_SIZE], BT_SRC_LANG, BT_PIVOT_LANG)
        )

    # Stage B: English -> Bengali (back)
    print("Stage B: English -> Bengali ...")
    bt_texts = []
    for i in range(0, len(en_texts), BT_BATCH_SIZE):
        bt_texts.extend(
            back_translate_batch(en_texts[i:i+BT_BATCH_SIZE], BT_PIVOT_LANG, BT_SRC_LANG)
        )

    # Build augmented rows — skip if back-translation is identical to original
    aug_rows = []
    for i, row in enumerate(eligible.itertuples()):
        bt_resp = normalize_text(bt_texts[i])
        if bt_resp and bt_resp != row.response_bn and len(bt_resp) > 5:
            aug_rows.append({
                "prompt_bn":   row.prompt_bn,
                "response_bn": bt_resp,
                "context":     row.context,
                "has_context": row.has_context,
                "label":       row.label,
            })

    del bt_mdl; gc.collect(); torch.cuda.empty_cache()

    if aug_rows:
        aug_df = pd.DataFrame(aug_rows)
        df_aug = pd.concat([df, aug_df], ignore_index=True)
        y_aug  = df_aug["label"].values
        cw_aug = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_aug)
        CLASS_WEIGHTS = torch.tensor(cw_aug, dtype=torch.float)
        print(f"\nAugmented training set: {len(df)} -> {len(df_aug)} rows ({len(aug_rows)} added)")
        print(f"Augmented class weights: 0={cw_aug[0]:.3f}  1={cw_aug[1]:.3f}")
    else:
        df_aug = df; y_aug = y
        print("Back-translation produced no new rows -- using original set.")
else:
    df_aug = df; y_aug = y
    print("Back-translation skipped -- using original training set.")

df_aug["nli_score"] = np.nan
df_aug["llm_score"] = 0.5
print(f"Training set for CV: {len(df_aug)} rows")

Loading NLLB-200: facebook/nllb-200-distilled-600M


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB-200 ready.
Eligible rows for back-translation: 299 / 299
Stage A: Bengali -> English ...
Stage B: English -> Bengali ...

Augmented training set: 299 -> 553 rows (254 added)
Augmented class weights: 0=1.106  1=0.913
Training set for CV: 553 rows


## 3. Dataset

**Input format** — explicit role markers help XLM-RoBERTa understand the task structure:
```
Context rows  : [CLS] Context: {ctx} [SEP] Question: {prompt} Answer: {response} [SEP]
No-context    : [CLS] Question: {prompt} [SEP] Answer: {response} [SEP]
```
Explicit `Context:` / `Question:` / `Answer:` prefixes act as soft task instructions
that prime the model on what to compare.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_MODEL)
print(f"Tokenizer loaded: {BACKBONE_MODEL}")

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: xlm-roberta-large


In [8]:
class HallucinationDataset(Dataset):
    def __init__(self, frame, tok, max_len, has_labels=True):
        self.frame      = frame.reset_index(drop=True)
        self.tok        = tok
        self.max_len    = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        # Explicit role markers improve task understanding for XLM-R
        if row["context"]:
            text_a = f"Context: {row['context']}"
            text_b = f"Question: {row['prompt_bn']} Answer: {row['response_bn']}"
        else:
            text_a = f"Question: {row['prompt_bn']}"
            text_b = f"Answer: {row['response_bn']}"

        enc = self.tok(
            text_a, text_b,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"].squeeze(0)
        if self.has_labels:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

## 4. 2-Stage Training Engine

```
Stage 1 — HEAD ONLY (encoder fully frozen)
  • Only 2,048 params train (classifier head)
  • LR = 3e-4  — safe to use high LR since encoder is frozen
  • 5 epochs  — teaches the head the correct output space
  • No catastrophic forgetting possible

Stage 2 — TOP 6 ENCODER LAYERS unfrozen
  • ~30M / 560M params train (top 6 of 24 layers)
  • LR = 8e-6  — very low to preserve pre-trained multilingual knowledge
  • 12 epochs with warmup and linear decay
  • Head is already well-initialized from Stage 1 → stable gradients
```

This is the standard best practice for fine-tuning large transformers on small datasets.

In [9]:
def freeze_all_except_head(model):
    for name, param in model.named_parameters():
        if "classifier" not in name and "pooler" not in name:
            param.requires_grad = False
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"    Frozen encoder | trainable: {n_train:,} / {n_total:,} ({100*n_train/n_total:.2f}%)")


def unfreeze_top_n(model, n):
    """Unfreeze top N encoder layers + pooler. Works for roberta/bert/deberta."""
    enc = None
    for attr in ["roberta", "bert", "deberta"]:
        if hasattr(model, attr):
            base = getattr(model, attr)
            if hasattr(base, "encoder") and hasattr(base.encoder, "layer"):
                enc = base.encoder.layer
                pooler = getattr(base, "pooler", None)
            break

    if enc is None:
        for p in model.parameters(): p.requires_grad = True
        print("    Unfroze ALL (could not detect encoder).")
        return

    total = len(enc)
    for i, layer in enumerate(enc):
        if i >= total - n:
            for p in layer.parameters(): p.requires_grad = True
    if pooler:
        for p in pooler.parameters(): p.requires_grad = True

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"    Unfroze top {n} layers | trainable: {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)")


def make_adamw(model, lr):
    no_decay = ["bias", "LayerNorm.weight"]
    return torch.optim.AdamW([
        {"params": [p for n,p in model.named_parameters() if p.requires_grad and not any(d in n for d in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n,p in model.named_parameters() if p.requires_grad and     any(d in n for d in no_decay)], "weight_decay": 0.0},
    ], lr=lr)


def run_epoch(model, loader, optimizer, scheduler, loss_fn, train):
    model.train() if train else model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            labels  = batch.pop("labels") if "labels" in batch else None
            logits  = model(**batch).logits
            if labels is not None:
                loss = loss_fn(logits, labels)
                if train:
                    optimizer.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    if scheduler: scheduler.step()
                total_loss += loss.item()
                all_labels.append(labels.cpu())
            all_logits.append(logits.detach().cpu())
    avg_loss  = total_loss / len(loader)
    probas    = torch.softmax(torch.cat(all_logits), dim=-1)[:, 1].numpy()
    labels_np = torch.cat(all_labels).numpy() if all_labels else None
    return avg_loss, probas, labels_np


def find_best_threshold(y_true, y_proba, steps=81):
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.10, 0.90, steps):
        s = f1_score(y_true, (y_proba >= t).astype(int), average="macro")
        if s > best_f1: best_f1, best_t = s, t
    return best_t, best_f1

print("Training utilities ready.")

Training utilities ready.


In [10]:
def train_one_fold(fold_idx, train_frame, val_frame):
    set_seed(RANDOM_SEED + fold_idx)

    # num_workers=0 fixes 'can only test a child process' on Kaggle
    tr_ds = HallucinationDataset(train_frame, tokenizer, MAX_LEN)
    vl_ds = HallucinationDataset(val_frame,   tokenizer, MAX_LEN)
    tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE,     shuffle=True,  num_workers=0, pin_memory=True)
    vl_ld = DataLoader(vl_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)

    model   = AutoModelForSequenceClassification.from_pretrained(
        BACKBONE_MODEL, num_labels=2, ignore_mismatched_sizes=True
    ).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))

    best_f1, best_val_probas = 0.0, None
    save_path = str(OUTPUT_DIR / f"fold_{fold_idx}.pt")

    # ── Stage 1: head only ─────────────────────────────────────────────────────
    print(f"  Stage 1 — head only (encoder frozen)")
    freeze_all_except_head(model)
    opt1 = make_adamw(model, S1_LR)

    for ep in range(1, S1_EPOCHS + 1):
        tl, _,  _       = run_epoch(model, tr_ld, opt1, None, loss_fn, True)
        _,  vp, vl      = run_epoch(model, vl_ld, opt1, None, loss_fn, False)
        vf1 = f1_score(vl, (vp >= 0.5).astype(int), average="macro")
        mark = " *** best" if vf1 > best_f1 else ""
        print(f"    Fold {fold_idx} S1 Ep{ep:2d} | loss={tl:.4f} | val_f1={vf1:.4f}{mark}")
        if vf1 > best_f1:
            best_f1, best_val_probas = vf1, vp.copy()
            torch.save(model.state_dict(), save_path)

    # ── Stage 2: top N encoder layers ─────────────────────────────────────────
    print(f"  Stage 2 — top {S2_UNFREEZE_LAST} encoder layers unfrozen")
    unfreeze_top_n(model, S2_UNFREEZE_LAST)
    opt2  = make_adamw(model, S2_LR)
    total = len(tr_ld) * S2_EPOCHS
    sched = get_linear_schedule_with_warmup(opt2, int(total * WARMUP_RATIO), total)

    for ep in range(1, S2_EPOCHS + 1):
        tl, _,  _       = run_epoch(model, tr_ld, opt2, sched, loss_fn, True)
        _,  vp, vl      = run_epoch(model, vl_ld, opt2, None,  loss_fn, False)
        vf1 = f1_score(vl, (vp >= 0.5).astype(int), average="macro")
        mark = " *** best" if vf1 > best_f1 else ""
        print(f"    Fold {fold_idx} S2 Ep{ep:2d} | loss={tl:.4f} | val_f1={vf1:.4f}{mark}")
        if vf1 > best_f1:
            best_f1, best_val_probas = vf1, vp.copy()
            torch.save(model.state_dict(), save_path)

    del model; torch.cuda.empty_cache()
    return best_f1, best_val_probas, save_path

## 5. K-Fold Cross-Validation Training

In [11]:
# Use augmented training set (df_aug includes back-translated rows if enabled)
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
oof_xlmr    = np.zeros(len(df_aug))
fold_f1s    = []
model_paths = []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(df_aug, y_aug), 1):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}  |  train={len(tr_idx)}  val={len(vl_idx)}")
    print(f"{'='*60}")
    best_f1, val_pr, mp = train_one_fold(fold, df_aug.iloc[tr_idx], df_aug.iloc[vl_idx])
    oof_xlmr[vl_idx]    = val_pr
    fold_f1s.append(best_f1)
    model_paths.append(mp)
    print(f"  Fold {fold} best val F1: {best_f1:.4f}")

print(f"\n{'='*60}")
print(f"XLM-RoBERTa CV ({N_FOLDS} folds):")
for i, s in enumerate(fold_f1s, 1): print(f"  Fold {i}: {s:.4f}")
print(f"  Mean : {np.mean(fold_f1s):.4f} \u00b1 {np.std(fold_f1s):.4f}")

# Alias for ensemble evaluation (matches augmented set size)
y_for_oof = y_aug


  FOLD 1/5  |  train=442  val=111


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Stage 1 — head only (encoder frozen)
    Frozen encoder | trainable: 1,051,650 / 559,892,482 (0.19%)
    Fold 1 S1 Ep 1 | loss=0.7312 | val_f1=0.3973 *** best
    Fold 1 S1 Ep 2 | loss=0.7493 | val_f1=0.5018 *** best
    Fold 1 S1 Ep 3 | loss=0.7374 | val_f1=0.3763
    Fold 1 S1 Ep 4 | loss=0.7095 | val_f1=0.5256 *** best
    Fold 1 S1 Ep 5 | loss=0.6854 | val_f1=0.5432 *** best
  Stage 2 — top 8 encoder layers unfrozen
    Unfroze top 8 layers | trainable: 101,821,442 / 559,892,482 (18.2%)
    Fold 1 S2 Ep 1 | loss=0.6785 | val_f1=0.5043
    Fold 1 S2 Ep 2 | loss=0.6676 | val_f1=0.5045
    Fold 1 S2 Ep 3 | loss=0.6650 | val_f1=0.5466 *** best
    Fold 1 S2 Ep 4 | loss=0.6140 | val_f1=0.6044 *** best
    Fold 1 S2 Ep 5 | loss=0.5942 | val_f1=0.4689
    Fold 1 S2 Ep 6 | loss=0.5094 | val_f1=0.6458 *** best
    Fold 1 S2 Ep 7 | loss=0.3728 | val_f1=0.6412
    Fold 1 S2 Ep 8 | loss=0.3003 | val_f1=0.7380 *** best
    Fold 1 S2 Ep 9 | loss=0.2145 | val_f1=0.6442
    Fold 1 S2 Ep10 | loss

...

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Stage 1 — head only (encoder frozen)
    Frozen encoder | trainable: 1,051,650 / 559,892,482 (0.19%)
    Fold 4 S1 Ep 1 | loss=0.7527 | val_f1=0.3529 *** best
    Fold 4 S1 Ep 2 | loss=0.6869 | val_f1=0.5238 *** best
    Fold 4 S1 Ep 3 | loss=0.7304 | val_f1=0.3125
    Fold 4 S1 Ep 4 | loss=0.7098 | val_f1=0.5487 *** best
    Fold 4 S1 Ep 5 | loss=0.6894 | val_f1=0.5408
  Stage 2 — top 8 encoder layers unfrozen
    Unfroze top 8 layers | trainable: 101,821,442 / 559,892,482 (18.2%)
    Fold 4 S2 Ep 1 | loss=0.6707 | val_f1=0.5113
    Fold 4 S2 Ep 2 | loss=0.6642 | val_f1=0.5832 *** best
    Fold 4 S2 Ep 3 | loss=0.6637 | val_f1=0.4500
    Fold 4 S2 Ep 4 | loss=0.6118 | val_f1=0.5635
    Fold 4 S2 Ep 5 | loss=0.5584 | val_f1=0.6404 *** best
    Fold 4 S2 Ep 6 | loss=0.5232 | val_f1=0.7314 *** best
    Fold 4 S2 Ep 7 | loss=0.3875 | val_f1=0.7629 *** best
    Fold 4 S2 Ep 8 | loss=0.2379 | val_f1=0.7128
    Fold 4 S2 Ep 9 | loss=0.1794 | val_f1=0.8089 *** best
    Fold 4 S2 Ep10 | loss

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Stage 1 — head only (encoder frozen)
    Frozen encoder | trainable: 1,051,650 / 559,892,482 (0.19%)
    Fold 5 S1 Ep 1 | loss=0.7473 | val_f1=0.3125 *** best
    Fold 5 S1 Ep 2 | loss=0.7016 | val_f1=0.5317 *** best
    Fold 5 S1 Ep 3 | loss=0.7042 | val_f1=0.4670
    Fold 5 S1 Ep 4 | loss=0.7125 | val_f1=0.5604 *** best
    Fold 5 S1 Ep 5 | loss=0.6928 | val_f1=0.4873
  Stage 2 — top 8 encoder layers unfrozen
    Unfroze top 8 layers | trainable: 101,821,442 / 559,892,482 (18.2%)
    Fold 5 S2 Ep 1 | loss=0.6843 | val_f1=0.6023 *** best
    Fold 5 S2 Ep 2 | loss=0.6613 | val_f1=0.6614 *** best
    Fold 5 S2 Ep 3 | loss=0.6547 | val_f1=0.6272
    Fold 5 S2 Ep 4 | loss=0.6182 | val_f1=0.6573
    Fold 5 S2 Ep 5 | loss=0.4685 | val_f1=0.6673 *** best
    Fold 5 S2 Ep 6 | loss=0.5053 | val_f1=0.7043 *** best
    Fold 5 S2 Ep 7 | loss=0.3432 | val_f1=0.7220 *** best
    Fold 5 S2 Ep 8 | loss=0.2042 | val_f1=0.7571 *** best
    Fold 5 S2 Ep 9 | loss=0.1549 | val_f1=0.7744 *** best
    Fol

## 6. NLI Entailment Signal (Context Rows)

**Why this matters:** The mDeBERTa NLI model achieves **0.73 accuracy** on context rows standalone.
Context rows are 43% of training data. Combining NLI with the transformer ensemble gives
a major boost on the open-book subset.

We score: does the `context` (premise) **entail** the `response_bn` (hypothesis)?
High entailment score → faithful; low score → hallucinated.

In [12]:
if not SKIP_NLI_SIGNAL:
    print(f"Loading NLI model: {NLI_MODEL}")
    nli_pipe = hf_pipeline(
        "zero-shot-classification", model=NLI_MODEL,
        device=0 if DEVICE == "cuda" else -1,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )

    def nli_score(ctx, resp):
        r = nli_pipe(ctx, candidate_labels=[resp], hypothesis_template="{}", multi_label=True)
        return r["scores"][0]

    # Training set
    df["nli_score"] = np.nan
    ctx_rows = df[df["has_context"]]
    scores   = [nli_score(r.context, r.response_bn)
                for r in tqdm(ctx_rows.itertuples(), total=len(ctx_rows), desc="NLI train")]
    df.loc[ctx_rows.index, "nli_score"] = scores

    # Test set
    test_df["nli_score"] = np.nan
    test_ctx = test_df[test_df["has_context"]]
    t_scores = [nli_score(r.context, r.response_bn)
                for r in tqdm(test_ctx.itertuples(), total=len(test_ctx), desc="NLI test")]
    test_df.loc[test_ctx.index, "nli_score"] = t_scores

    # Sanity check
    ctx_eval = df.loc[df["has_context"]].dropna(subset=["nli_score"])
    nli_pred = (ctx_eval["nli_score"] > 0.5).astype(int)
    nli_f1   = f1_score(ctx_eval["label"], nli_pred, average="macro")
    print(f"\nNLI standalone on context rows — Macro-F1: {nli_f1:.4f}")
    del nli_pipe; torch.cuda.empty_cache()
    # Copy NLI scores to df_aug (augmented rows get neutral 0.5)
    df_aug.loc[df.index, "nli_score"] = df["nli_score"].values
    df_aug["nli_score"] = df_aug["nli_score"].fillna(0.5)
else:
    df["nli_score"]      = np.nan
    df_aug["nli_score"]  = 0.5
    test_df["nli_score"] = np.nan
    print("NLI signal skipped.")

Loading NLI model: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

NLI train:   0%|          | 0/130 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


NLI test:   0%|          | 0/1361 [00:00<?, ?it/s]


NLI standalone on context rows — Macro-F1: 0.7281


## 7. LLM Judge — Qwen2.5-7B-Instruct (All Rows)

**Why this is critical:** 57% of rows have NO context. For those, the transformer and NLI
cannot check factual correctness — only a model with world knowledge can.

The LLM judge uses Qwen2.5-7B-Instruct with 4-bit NF4 quantization (fits in T4 VRAM).
It runs on ALL rows, providing a strong signal for closed-book factual QA.

Expected contribution: +5–8 macro-F1 points on top of the transformer baseline.

In [13]:
if not SKIP_LLM_JUDGE:
    import gc
    from transformers import AutoModelForCausalLM, AutoTokenizer as AT, BitsAndBytesConfig

    # ── Aggressive VRAM cleanup before loading 7B model ──────────────────────
    gc.collect()
    torch.cuda.empty_cache()
    if DEVICE == "cuda":
        print(f"VRAM free before LLM load: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

    print(f"Loading LLM judge: {LLM_MODEL} (4-bit NF4)")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    llm_tok = AT.from_pretrained(LLM_MODEL)
    llm_mdl = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, quantization_config=bnb, device_map="auto"
    )
    llm_mdl.eval()

    # Pre-compute token ids for "0" and "1" answers
    _tok_0 = llm_tok.encode("0", add_special_tokens=False)[0]
    _tok_1 = llm_tok.encode("1", add_special_tokens=False)[0]
    print(f"LLM ready.  token_id for 0={_tok_0}, 1={_tok_1}")

    # ── OOM-safe prompt truncation constants ─────────────────────────────────
    # 768 tokens max keeps peak VRAM under 4 GB for 4-bit Qwen2.5-7B on T4.
    # Long context rows are truncated from the LEFT (drops early context, keeps Q+A).
    LLM_MAX_TOKENS = 768

    JUDGE_SYS = (
        "You are a strict Bengali fact-checker. "
        "Given a question and an answer (and optionally a reference context), "
        "decide if the answer is factually correct. "
        "If a context is given, judge only based on the context. "
        "If no context, use your general knowledge. "
        "Reply with ONLY the digit 1 (correct/faithful) or 0 (wrong/hallucinated). Nothing else."
    )

    def build_judge_prompt(prompt_bn, response_bn, context):
        if context:
            u = f"Context: {context}\nQuestion: {prompt_bn}\nAnswer: {response_bn}\n\nCorrect (1) or Wrong (0)?"
        else:
            u = f"Question: {prompt_bn}\nAnswer: {response_bn}\n\nCorrect (1) or Wrong (0)?"
        return [{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": u}]

    def judge_score(prompt_bn, response_bn, context):
        """
        OOM-safe logits-based scorer.
        Instead of calling .generate() (which builds a KV cache and OOMs on long inputs),
        we do a single forward pass and read the next-token logits for tokens "0" and "1".
        This uses ~3-4x less VRAM than generation and is much faster.
        """
        msgs = build_judge_prompt(prompt_bn, response_bn, context)
        txt  = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        # Truncate from the LEFT so the Q+A is always preserved at the end
        inp  = llm_tok(
            txt, return_tensors="pt",
            truncation=True, max_length=LLM_MAX_TOKENS,
        ).to(llm_mdl.device)

        try:
            with torch.no_grad():
                logits = llm_mdl(**inp).logits   # (1, seq_len, vocab)
            # Next-token logits at the last position
            last_logits = logits[0, -1, :]
            # Softmax only over "0" and "1" — probability the answer is faithful
            pair  = last_logits[[_tok_0, _tok_1]].float()
            proba = torch.softmax(pair, dim=0)[1].item()
            return proba
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            return 0.5   # neutral fallback on OOM

    # ── Run on training set ───────────────────────────────────────────────────
    train_scores = []
    for r in tqdm(df.itertuples(), total=len(df), desc="LLM judge train"):
        train_scores.append(judge_score(r.prompt_bn, r.response_bn, r.context))
    df["llm_score"] = train_scores

    # ── Run on test set (main OOM site — now safe) ────────────────────────────
    test_scores = []
    for i, r in enumerate(tqdm(test_df.itertuples(), total=len(test_df), desc="LLM judge test")):
        test_scores.append(judge_score(r.prompt_bn, r.response_bn, r.context))
        # Periodic cache flush every 200 rows to prevent fragmentation
        if (i + 1) % 200 == 0:
            torch.cuda.empty_cache()
    test_df["llm_score"] = test_scores

    # ── Sanity check ──────────────────────────────────────────────────────────
    llm_f1 = f1_score(y, (df["llm_score"] > 0.5).astype(int), average="macro")
    print(f"\nLLM judge standalone — Train Macro-F1: {llm_f1:.4f}")
    print(f"LLM score distribution: mean={df['llm_score'].mean():.3f}  std={df['llm_score'].std():.3f}")

    del llm_mdl; gc.collect(); torch.cuda.empty_cache()

    # ── Propagate to df_aug (augmented rows inherit original LLM score) ───────
    df_aug.loc[df.index, "llm_score"] = df["llm_score"].values
    df_aug["llm_score"] = df_aug["llm_score"].fillna(0.5)
else:
    df["llm_score"]      = 0.5
    df_aug["llm_score"]  = 0.5
    test_df["llm_score"] = 0.5
    print("LLM judge skipped.")

VRAM free before LLM load: 15.62 GB
Loading LLM judge: Qwen/Qwen2.5-7B-Instruct (4-bit NF4)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM ready.  token_id for 0=15, 1=16


LLM judge train:   0%|          | 0/299 [00:00<?, ?it/s]

LLM judge test:   0%|          | 0/2516 [00:00<?, ?it/s]


LLM judge standalone — Train Macro-F1: 0.6488
LLM score distribution: mean=0.460  std=0.436


## 8. Context-Aware Ensemble

Different signals are reliable for different row types:

| Row type | XLM-RoBERTa | NLI score | LLM judge |
|---|---|---|---|
| Has context | Strong | **Very strong** (0.73 acc standalone) | Moderate |
| No context | Moderate | Not applicable | **Very strong** |

We use a **split ensemble** that applies different weights per subset:
- Context rows   : 50% transformer + 35% NLI + 15% LLM
- No-context rows: 45% transformer + 0% NLI + 55% LLM

Then we optimize the decision threshold separately for each subset.

In [14]:
# ── Ensemble weights ──────────────────────────────────────────────────────────
# Adjust these if one signal appears stronger on your local CV results
W_XLMR_CTX  = 0.50   # transformer weight for context rows
W_NLI_CTX   = 0.35   # NLI weight for context rows
W_LLM_CTX   = 0.15   # LLM weight for context rows

W_XLMR_NCTX = 0.45   # transformer weight for no-context rows
W_LLM_NCTX  = 0.55   # LLM weight for no-context rows

# If a signal is skipped, redistribute its weight to transformer
if SKIP_NLI_SIGNAL:
    W_XLMR_CTX += W_NLI_CTX; W_NLI_CTX = 0.0
if SKIP_LLM_JUDGE:
    W_XLMR_CTX += W_LLM_CTX; W_LLM_CTX = 0.0
    W_XLMR_NCTX += W_LLM_NCTX; W_LLM_NCTX = 0.0

def blend(xlmr_pr, nli_pr, llm_pr, has_ctx_mask):
    out = np.zeros(len(xlmr_pr))
    c   = has_ctx_mask
    nc  = ~has_ctx_mask
    nli_safe = np.where(np.isnan(nli_pr), 0.5, nli_pr)

    if c.any():
        out[c]  = W_XLMR_CTX  * xlmr_pr[c] + W_NLI_CTX  * nli_safe[c] + W_LLM_CTX  * llm_pr[c]
    if nc.any():
        out[nc] = W_XLMR_NCTX * xlmr_pr[nc]                             + W_LLM_NCTX * llm_pr[nc]
    return out

# ── OOF ensemble probabilities ────────────────────────────────────────────────
oof_ensemble = blend(
    oof_xlmr,
    df_aug["nli_score"].values,
    df_aug["llm_score"].values,
    df_aug["has_context"].values,
)

# ── Evaluate at default threshold ─────────────────────────────────────────────
oof_pred_05 = (oof_ensemble >= 0.5).astype(int)
oof_f1_05   = f1_score(y_for_oof, oof_pred_05, average="macro")
print(f"OOF Ensemble Macro-F1 @ 0.50 : {oof_f1_05:.4f}")
print()
print(classification_report(y_for_oof, oof_pred_05, target_names=["hallucinated(0)", "faithful(1)"]))

OOF Ensemble Macro-F1 @ 0.50 : 0.7890

                 precision    recall  f1-score   support

hallucinated(0)       0.76      0.79      0.77       250
    faithful(1)       0.82      0.79      0.81       303

       accuracy                           0.79       553
      macro avg       0.79      0.79      0.79       553
   weighted avg       0.79      0.79      0.79       553



In [15]:
BEST_T, BEST_F1 = find_best_threshold(y_for_oof, oof_ensemble)
oof_pred_opt    = (oof_ensemble >= BEST_T).astype(int)

print(f"Optimal threshold              : {BEST_T:.2f}")
print(f"OOF Macro-F1 @ optimal         : {BEST_F1:.4f}")
print(f"Gain over default (0.50)       : +{BEST_F1 - oof_f1_05:.4f}")
print()
print(classification_report(y_for_oof, oof_pred_opt, target_names=["hallucinated(0)", "faithful(1)"]))

# Per-subset analysis
for name, mask in [("Context rows", df_aug["has_context"]), ("No-context rows", ~df_aug["has_context"])]:
    if mask.any():
        sub_f1 = f1_score(y_for_oof[mask], oof_pred_opt[mask], average="macro")
        print(f"  {name:16s} Macro-F1: {sub_f1:.4f}  (n={mask.sum()})")

Optimal threshold              : 0.43
OOF Macro-F1 @ optimal         : 0.8102
Gain over default (0.50)       : +0.0212

                 precision    recall  f1-score   support

hallucinated(0)       0.85      0.72      0.78       250
    faithful(1)       0.79      0.90      0.84       303

       accuracy                           0.82       553
      macro avg       0.82      0.81      0.81       553
   weighted avg       0.82      0.82      0.81       553

  Context rows     Macro-F1: 0.8664  (n=247)
  No-context rows  Macro-F1: 0.7613  (n=306)


## 9. Test Set Prediction

In [16]:
test_ds  = HallucinationDataset(test_df, tokenizer, MAX_LEN, has_labels=False)
test_ld  = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)

test_xlmr_list = []

for fold, mp in enumerate(model_paths, 1):
    print(f"Inference — fold {fold}")
    model = AutoModelForSequenceClassification.from_pretrained(
        BACKBONE_MODEL, num_labels=2, ignore_mismatched_sizes=True
    ).to(DEVICE)
    model.load_state_dict(torch.load(mp, map_location=DEVICE))
    model.eval()

    logits_list = []
    with torch.no_grad():
        for batch in tqdm(test_ld, desc=f"Fold {fold}", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits_list.append(model(**batch).logits.cpu())

    fold_pr = torch.softmax(torch.cat(logits_list), dim=-1)[:, 1].numpy()
    test_xlmr_list.append(fold_pr)
    del model; torch.cuda.empty_cache()

test_xlmr = np.mean(test_xlmr_list, axis=0)

# Apply context-aware ensemble to test
test_ensemble = blend(
    test_xlmr,
    test_df["nli_score"].values,
    test_df["llm_score"].values,
    test_df["has_context"].values,
)

final_preds = (test_ensemble >= BEST_T).astype(int)
print(f"Predicted label=1 rate: {final_preds.mean():.3f}")

Inference — fold 1


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 1:   0%|          | 0/158 [00:00<?, ?it/s]

Inference — fold 2


...

Inference — fold 5


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 5:   0%|          | 0/158 [00:00<?, ?it/s]

Predicted label=1 rate: 0.669


## 9b. Pseudo-Labeling: Round-2 Retraining

After Round 1 predictions, select test rows where the ensemble is **very confident** (proba >= 0.92
or <= 0.08). These become pseudo-labeled training examples.

We do a **quick Round-2** Stage-2 fine-tune on original + pseudo-labeled data using the same
model checkpoints. Only Stage 2 (top encoder layers) runs — Stage 1 head is already initialized.

> Expected gain: +4-7 macro-F1 points when the initial model has reasonable calibration.

In [17]:
if RUN_PSEUDO_LABEL:
    high_conf_mask = (test_ensemble >= PSEUDO_THRESH) | (test_ensemble <= (1 - PSEUDO_THRESH))
    n_pseudo = int(high_conf_mask.sum())
    print(f"High-confidence pseudo-labels: {n_pseudo} / {len(test_df)} test rows ({100*high_conf_mask.mean():.1f}%)")

    if n_pseudo < 30:
        print("Too few pseudo-labels (<30) -- skipping Round 2.")
        final_xlmr = test_xlmr
    else:
        pseudo_df = test_df[high_conf_mask].copy()
        pseudo_df = pseudo_df.reset_index(drop=True)
        pseudo_df["label"] = (test_ensemble[high_conf_mask] >= 0.5).astype(int)
        print(f"  label=1: {pseudo_df['label'].sum()}  label=0: {(pseudo_df['label']==0).sum()}")

        # Add required columns so concat works
        for col in ["nli_score", "llm_score"]:
            if col not in pseudo_df.columns:
                pseudo_df[col] = 0.5

        keep_cols = ["prompt_bn", "response_bn", "context", "has_context", "label", "nli_score", "llm_score"]
        df_r2 = pd.concat([df_aug[keep_cols], pseudo_df[keep_cols]], ignore_index=True)
        y_r2  = df_r2["label"].values
        cw_r2 = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_r2)
        CW_R2 = torch.tensor(cw_r2, dtype=torch.float)
        print(f"Round-2 training set: {len(df_r2)} rows (original {len(df_aug)} + pseudo {n_pseudo})")

        r2_test_preds = []
        for fold, mp in enumerate(model_paths, 1):
            print(f"\nRound-2 fold {fold} -- Stage 2 continue from checkpoint")
            model = AutoModelForSequenceClassification.from_pretrained(
                BACKBONE_MODEL, num_labels=2, ignore_mismatched_sizes=True
            ).to(DEVICE)
            model.load_state_dict(torch.load(mp, map_location=DEVICE))
            unfreeze_top_n(model, S2_UNFREEZE_LAST)

            tr_ds_r2 = HallucinationDataset(df_r2, tokenizer, MAX_LEN)
            tr_ld_r2 = DataLoader(tr_ds_r2, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
            loss_fn_r2 = nn.CrossEntropyLoss(weight=CW_R2.to(DEVICE))
            opt_r2 = make_adamw(model, S2_LR * 0.5)   # half LR for refinement
            total_r2 = len(tr_ld_r2) * PSEUDO_S2_EPOCHS
            sched_r2 = get_linear_schedule_with_warmup(opt_r2, int(total_r2 * WARMUP_RATIO), total_r2)

            for ep in range(1, PSEUDO_S2_EPOCHS + 1):
                tl, _, _ = run_epoch(model, tr_ld_r2, opt_r2, sched_r2, loss_fn_r2, True)
                print(f"    R2 Fold {fold} Ep{ep:2d} | train_loss={tl:.4f}")

            # Test inference with round-2 model
            test_ds_r2 = HallucinationDataset(test_df, tokenizer, MAX_LEN, has_labels=False)
            test_ld_r2 = DataLoader(test_ds_r2, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=True)
            logits_r2 = []
            with torch.no_grad():
                for batch in tqdm(test_ld_r2, desc=f"R2 Fold {fold} test", leave=False):
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    logits_r2.append(model(**batch).logits.cpu())
            fold_pr_r2 = torch.softmax(torch.cat(logits_r2), dim=-1)[:, 1].numpy()
            r2_test_preds.append(fold_pr_r2)

            torch.save(model.state_dict(), mp)
            del model; torch.cuda.empty_cache()

        test_xlmr_r2 = np.mean(r2_test_preds, axis=0)
        # Blend Round1 (50%) and Round2 (50%) XLM-R predictions
        final_xlmr = 0.5 * test_xlmr + 0.5 * test_xlmr_r2
        print(f"\nFinal XLM-R = 0.5 * Round1 + 0.5 * Round2")
        print(f"Round1 mean proba: {test_xlmr.mean():.3f}  Round2: {test_xlmr_r2.mean():.3f}")
else:
    final_xlmr = test_xlmr
    print("Pseudo-labeling skipped -- using Round 1 predictions.")

# Recompute final ensemble with (possibly improved) final_xlmr
test_ensemble_final = blend(
    final_xlmr,
    test_df["nli_score"].values,
    test_df["llm_score"].values,
    test_df["has_context"].values,
)
final_preds = (test_ensemble_final >= BEST_T).astype(int)
print(f"\nFinal ensemble label=1 rate: {final_preds.mean():.3f}")

High-confidence pseudo-labels: 421 / 2516 test rows (16.7%)
  label=1: 259  label=0: 162
Round-2 training set: 974 rows (original 553 + pseudo 421)

Round-2 fold 1 -- Stage 2 continue from checkpoint


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    Unfroze top 8 layers | trainable: 559,892,482 / 559,892,482 (100.0%)
    R2 Fold 1 Ep 1 | train_loss=0.2409
    R2 Fold 1 Ep 2 | train_loss=0.1605
    R2 Fold 1 Ep 3 | train_loss=0.0833
    R2 Fold 1 Ep 4 | train_loss=0.0849
    R2 Fold 1 Ep 5 | train_loss=0.0290
    R2 Fold 1 Ep 6 | train_loss=0.0380


R2 Fold 1 test:   0%|          | 0/158 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    Unfroze top 8 layers | trainable: 559,892,482 / 559,892,482 (100.0%)
    R2 Fold 5 Ep 1 | train_loss=0.2535
    R2 Fold 5 Ep 2 | train_loss=0.2251
    R2 Fold 5 Ep 3 | train_loss=0.1160
    R2 Fold 5 Ep 4 | train_loss=0.0584
    R2 Fold 5 Ep 5 | train_loss=0.0588
    R2 Fold 5 Ep 6 | train_loss=0.0380


R2 Fold 5 test:   0%|          | 0/158 [00:00<?, ?it/s]


Final XLM-R = 0.5 * Round1 + 0.5 * Round2
Round1 mean proba: 0.632  Round2: 0.636

Final ensemble label=1 rate: 0.674


In [18]:
submission = pd.DataFrame({"id": test_df["id"], "label": final_preds})
submission.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv — {len(submission)} rows, threshold={BEST_T:.2f}")
submission.head(10)

Wrote submission.csv — 2516 rows, threshold=0.43


,id,label
0,1,0
1,2,0
2,3,1
3,4,1
4,5,1
5,6,0
6,7,0
7,8,0
8,9,0
9,10,0


In [19]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission)          == len(sample_sub)
assert set(submission["id"])    == set(sample_sub["id"])
assert submission["label"].isin([0, 1]).all()
assert submission["label"].isna().sum() == 0

print("All sanity checks passed.\n")
print("Label distribution:")
print(submission["label"].value_counts(normalize=True).rename("share").to_frame())
print()
print(f"\n{'='*50}")
print(f"FINAL CV SUMMARY")
print(f"{'='*50}")
print(f"XLM-RoBERTa fold scores : {[round(s,4) for s in fold_f1s]}")
print(f"XLM-RoBERTa mean        : {np.mean(fold_f1s):.4f}")
print(f"Ensemble OOF @ t=0.50   : {oof_f1_05:.4f}")
print(f"Ensemble OOF @ t={BEST_T:.2f}   : {BEST_F1:.4f}  <-- use this")
if RUN_PSEUDO_LABEL:
    print(f"Pseudo-labeling       : enabled (thresh={PSEUDO_THRESH})")
if RUN_BACKTRANSLATION:
    print(f"Back-translation      : enabled ({len(df_aug)-len(df)} rows added)")

All sanity checks passed.

Label distribution:
          share
label          
1      0.673688
0      0.326312


FINAL CV SUMMARY
XLM-RoBERTa fold scores : [0.7648, 0.8886, 0.8638, 0.8623, 0.7932]
XLM-RoBERTa mean        : 0.8345
Ensemble OOF @ t=0.50   : 0.7890
Ensemble OOF @ t=0.43   : 0.8102  <-- use this
Pseudo-labeling       : enabled (thresh=0.92)
Back-translation      : enabled (254 rows added)


## 10. Troubleshooting & Further Improvements

### Quick config levers
| Setting | Default | Effect |
|---|---|---|
| `S2_EPOCHS` | 18 | More epochs -> better convergence on small data |
| `S2_UNFREEZE_LAST` | 8 | More unfrozen layers -> more task-specific adaptation |
| `PSEUDO_THRESH` | 0.92 | Lower -> more pseudo-labels (noisier) |
| `BT_MAX_CHARS` | 350 | Higher -> more rows augmented (lower quality) |

### If val F1 is below 0.65 after Stage 2
- Lower `S2_LR` from 8e-6 -> 5e-6 (less catastrophic forgetting)
- Try `BACKBONE_MODEL = 'xlm-roberta-base'` — sometimes -large overfits more on tiny data
- Set `RUN_BACKTRANSLATION = False` to isolate its effect on CV

### If ensemble OOF is between 0.65–0.75
- Tune `W_XLMR_CTX`, `W_NLI_CTX`, `W_LLM_CTX` based on per-signal standalone F1
- Enable both NLI and LLM judge if either was skipped
- Increase `MAX_LEN = 384` if context rows are truncated heavily

### To push past 0.82
- **Multi-task pretraining**: first fine-tune on IndicXNLI Bengali, then on this task
- **Better LLM prompting**: add 3-5 few-shot examples to the judge prompt
- **Calibration**: temperature scaling on XLM-R probabilities before blending
- **BanglaBERT ensemble**: add as 4th signal (~110M params, cheap to run)
